In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# --- Project root (robust to folder moves) ---
# Default to the known location; fall back to the current working directory
# if the project has been moved again (launch JupyterLab from the project folder).
PROJECT_ROOT = Path.home() / "Claude" / "Projects" / "Directed Study" / "Ransomware"
if not (PROJECT_ROOT / "features").exists():
    PROJECT_ROOT = Path.cwd()

# Paths
DATA_ROOT   = PROJECT_ROOT / "dataset"
FEATURES_DIR = PROJECT_ROOT / "features"

# Paper parameters
T_WINDOW = 0.1
T_D      = 30.0
EPSILON  = T_WINDOW

# Memory access type codes
TYPE_4KB  = 1
TYPE_2MB  = 2
TYPE_MMIO = 4

# Class definitions
RANSOMWARE_CLASSES   = ["WannaCry", "Ryuk", "REvil", "LockBit", "Darkside", "Conti"]
BENIGN_CLASSES       = ["AESCrypt", "Zip", "SDelete", "Firefox", "Office", "Idle"]
ALL_ORIGINAL_CLASSES = RANSOMWARE_CLASSES + BENIGN_CLASSES
VARIANT_CLASSES      = [f"Conti_{i:02d}" for i in range(1, 8)]

print("Config loaded.")
print(f"Data root:    {DATA_ROOT}")
print(f"Features dir: {FEATURES_DIR}")

Config loaded.
Data root:    /home/arshad/Claude/Projects/Directed Study/Ransomware/dataset
Features dir: /home/arshad/Claude/Projects/Directed Study/Ransomware/features


In [11]:
def load_csv_safe(path):
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path, header=None)
    except Exception as e:
        print(f"  Warning: could not read {path}: {e}")
        return pd.DataFrame()

def calc_throughput(lines):
    if lines.empty:
        return 0.0
    return float(lines.iloc[:, 3].sum()) / T_WINDOW

def calc_LBA_variance(lines):
    if len(lines) < 2:
        return 0.0
    return float(lines.iloc[:, 2].astype(np.float64).var(ddof=1))

def calc_shannon_entropy(lines):
    if lines.empty:
        return 0.0
    valid = lines.iloc[:, 4][lines.iloc[:, 4] >= 0.0]
    return float(valid.mean()) if not valid.empty else 0.0

def calc_memory_entropy(lines):
    if lines.empty:
        return 0.0
    valid = lines.iloc[:, 4][lines.iloc[:, 4] > 0.0]
    return float(valid.mean()) if not valid.empty else 0.0

def count_page_access(lines, page_type):
    if lines.empty:
        return 0
    return int((lines.iloc[:, -1] == page_type).sum())

def calc_GPA_variance(lines):
    if len(lines) < 2:
        return 0.0
    return float(lines.iloc[:, 2].astype(np.float64).var(ddof=1))

def add_relative_timestamps(df, ref_ns):
    df = df.copy()
    t = df.iloc[:, 0].astype(np.int64) * 1_000_000_000 + df.iloc[:, 1].astype(np.int64)
    df["rel_ns"] = t - ref_ns
    return df

def get_ref_ns(frames):
    ref_ns = None
    for df in frames:
        if not df.empty:
            t0 = df.iloc[0, 0] * 1_000_000_000 + df.iloc[0, 1]
            if ref_ns is None or t0 < ref_ns:
                ref_ns = t0
    return ref_ns

print("Helper functions loaded.")

Helper functions loaded.


In [12]:
def calculate_storage_feature(exec_dir):
    lines_sr = load_csv_safe(exec_dir / "ata_read.csv")
    lines_sw = load_csv_safe(exec_dir / "ata_write.csv")

    if lines_sr.empty and lines_sw.empty:
        return np.zeros((0, 5), dtype=np.float32)

    ref_ns = get_ref_ns([lines_sr, lines_sw])
    if not lines_sr.empty:
        lines_sr = add_relative_timestamps(lines_sr, ref_ns)
    if not lines_sw.empty:
        lines_sw = add_relative_timestamps(lines_sw, ref_ns)

    T_window_ns = int(T_WINDOW * 1_000_000_000)
    T_d_ns      = int(T_D      * 1_000_000_000)
    epsilon_ns  = int(EPSILON  * 1_000_000_000)

    chi_s = []
    t = 0
    while t <= T_d_ns - T_window_ns:
        t_end = t + T_window_ns

        def win(df):
            if df.empty: return df
            return df[(df["rel_ns"] >= t) & (df["rel_ns"] < t_end)]

        w_sr = win(lines_sr)
        w_sw = win(lines_sw)

        chi_s.append([
            calc_throughput(w_sr),
            calc_throughput(w_sw),
            calc_LBA_variance(w_sr),
            calc_LBA_variance(w_sw),
            calc_shannon_entropy(w_sw),
        ])
        t += epsilon_ns

    return np.array(chi_s, dtype=np.float32)

print("Storage feature function loaded.")

Storage feature function loaded.


In [13]:
def calculate_memory_feature(exec_dir):
    lines_mr  = load_csv_safe(exec_dir / "mem_read.csv")
    lines_mw  = load_csv_safe(exec_dir / "mem_write.csv")
    lines_mrw = load_csv_safe(exec_dir / "mem_readwrite.csv")
    lines_mx  = load_csv_safe(exec_dir / "mem_exec.csv")

    all_frames = [lines_mr, lines_mw, lines_mrw, lines_mx]
    ref_ns = get_ref_ns(all_frames)
    if ref_ns is None:
        return np.zeros((0, 18), dtype=np.float32)

    lines_mr, lines_mw, lines_mrw, lines_mx = [
        add_relative_timestamps(df, ref_ns) if not df.empty else df
        for df in all_frames
    ]

    T_window_ns = int(T_WINDOW * 1_000_000_000)
    T_d_ns      = int(T_D      * 1_000_000_000)
    epsilon_ns  = int(EPSILON  * 1_000_000_000)

    chi_m = []
    t = 0
    while t <= T_d_ns - T_window_ns:
        t_end = t + T_window_ns

        def win(df):
            if df.empty: return df
            return df[(df["rel_ns"] >= t) & (df["rel_ns"] < t_end)]

        w_mr  = win(lines_mr)
        w_mw  = win(lines_mw)
        w_mrw = win(lines_mrw)
        w_mx  = win(lines_mx)

        chi_m.append([
            calc_memory_entropy(w_mw),
            calc_memory_entropy(w_mrw),
            count_page_access(w_mr,  TYPE_4KB),
            count_page_access(w_mw,  TYPE_4KB),
            count_page_access(w_mrw, TYPE_4KB),
            count_page_access(w_mx,  TYPE_4KB),
            count_page_access(w_mr,  TYPE_2MB),
            count_page_access(w_mw,  TYPE_2MB),
            count_page_access(w_mrw, TYPE_2MB),
            count_page_access(w_mx,  TYPE_2MB),
            count_page_access(w_mr,  TYPE_MMIO),
            count_page_access(w_mw,  TYPE_MMIO),
            count_page_access(w_mrw, TYPE_MMIO),
            count_page_access(w_mx,  TYPE_MMIO),
            calc_GPA_variance(w_mr),
            calc_GPA_variance(w_mw),
            calc_GPA_variance(w_mrw),
            calc_GPA_variance(w_mx),
        ])
        t += epsilon_ns

    return np.array(chi_m, dtype=np.float32)

print("Memory feature function loaded.")

Memory feature function loaded.


In [14]:
def extract_features(exec_dir):
    chi_s = calculate_storage_feature(exec_dir)
    chi_m = calculate_memory_feature(exec_dir)
    if chi_s.shape[0] == 0 or chi_m.shape[0] == 0:
        return None
    n = min(chi_s.shape[0], chi_m.shape[0])
    return np.concatenate([chi_s[:n], chi_m[:n]], axis=1)

def scan_original(data_root):
    records = []
    class_dir = data_root / "ddr4-2133-16g"
    for class_name in ALL_ORIGINAL_CLASSES:
        class_path = class_dir / class_name
        if not class_path.exists():
            print(f"  Warning: {class_path} not found")
            continue
        for exec_dir in sorted(d for d in class_path.iterdir() if d.is_dir()):
            records.append({"exec_dir": exec_dir, "label": class_name, "group": "original"})
    return records

def scan_variants(data_root):
    records = []
    variants_dir = data_root / "variants" / "i3-gen12" / "ddr4-2133-16g"
    for variant_name in VARIANT_CLASSES:
        variant_path = variants_dir / variant_name
        if not variant_path.exists():
            print(f"  Warning: {variant_path} not found")
            continue
        for exec_dir in sorted(d for d in variant_path.iterdir() if d.is_dir()):
            records.append({"exec_dir": exec_dir, "label": variant_name, "group": "variant"})
    return records

print("Scanning functions loaded.")

Scanning functions loaded.


In [15]:
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

records = scan_original(DATA_ROOT) + scan_variants(DATA_ROOT)
print(f"Found {len(records)} execution directories.")

metadata_rows = []
success, failed = 0, 0

for i, rec in enumerate(records):
    exec_dir = rec["exec_dir"]
    label    = rec["label"]
    group    = rec["group"]

    rel_path = exec_dir.relative_to(DATA_ROOT)
    out_name = str(rel_path).replace("/", "__").replace("\\", "__") + ".npy"
    out_path = FEATURES_DIR / out_name

    if out_path.exists():
        metadata_rows.append({"filepath": str(out_path), "label": label,
                               "group": group, "exec_dir": str(exec_dir)})
        success += 1
        continue

    features = extract_features(exec_dir)
    if features is None:
        print(f"FAILED: {exec_dir.name}")
        failed += 1
        continue

    np.save(out_path, features)
    metadata_rows.append({"filepath": str(out_path), "label": label,
                           "group": group, "exec_dir": str(exec_dir)})
    success += 1

meta_df = pd.DataFrame(metadata_rows)
meta_df.to_csv(FEATURES_DIR / "metadata.csv", index=False)
print(f"\nDone. Success: {success}  Failed: {failed}")

Found 190 execution directories.

Done. Success: 190  Failed: 0


In [16]:
check_labels = ["WannaCry", "Conti", "REvil", "LockBit", "Darkside", "Ryuk",
                "AESCrypt", "SDelete", "Zip", "Firefox", "Office", "Idle"]

print(f"{'Label':<12}  {'Entropy Mean':>12}  {'Entropy Max':>11}  {'Type'}")
print("-" * 50)

for label in check_labels:
    files = meta_df[meta_df["label"] == label]["filepath"].values
    if len(files) == 0:
        continue
    # Average entropy across all executions for this class
    entropies = [np.load(f)[:, 4].mean() for f in files]
    mean_ent  = np.mean(entropies)
    max_ent   = np.max([np.load(f)[:, 4].max() for f in files])
    label_type = "RANSOMWARE" if label in RANSOMWARE_CLASSES else "BENIGN"
    print(f"{label:<12}  {mean_ent:>12.4f}  {max_ent:>11.4f}  {label_type}")

Label         Entropy Mean  Entropy Max  Type
--------------------------------------------------
WannaCry            0.2582       0.9946  RANSOMWARE
Conti               0.1363       0.9942  RANSOMWARE
REvil               0.0649       0.9533  RANSOMWARE
LockBit             0.0989       0.9946  RANSOMWARE
Darkside            0.1387       0.9592  RANSOMWARE
Ryuk                0.0548       0.9834  RANSOMWARE
AESCrypt            0.1872       0.9945  BENIGN
SDelete             0.1449       0.9944  BENIGN
Zip                 0.1380       0.9944  BENIGN
Firefox             0.0988       0.9944  BENIGN
Office              0.0619       0.9756  BENIGN
Idle                0.0614       0.9941  BENIGN
